# Chapter 2: End-to-End Machine Learning Project

**Concepts & Theory Reference:** [`Concepts.md`](../Concepts.md) — detailed explanations of every concept used in this notebook (stratified sampling, pipelines, RMSE, cross-validation, overfitting, ensembles, hyperparameter tuning, feature importances, test set evaluation).

---

## Phase 1: Data — "What do we have?"
*Load the raw dataset, create a representative train/test split, and separate features from the target variable. Everything downstream depends on this foundation being right.*

In [1]:
# =============================================================================
# CHAPTER 2: END-TO-END MACHINE LEARNING PROJECT — Setup & Data Split
# =============================================================================
#
# GOAL: Predict median house value for California districts using census data.
#
# WORKFLOW:
#   Cell 0 → Load data, stratified train/test split, separate features/labels
#   Cell 1 → Full preprocessing pipeline (24 engineered features)
#   Cell 2 → Train LinearRegression baseline, cross-validated RMSE
# =============================================================================

from pathlib import Path
import pandas as pd
import numpy as np
import tarfile
import urllib.request

# sklearn — data splitting & evaluation
from sklearn.model_selection import train_test_split, cross_val_score

# sklearn — imputation & encoding
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer

# sklearn — pipeline & composition
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer, make_column_selector

# sklearn — clustering & similarity (used in ClusterSimilarity)
from sklearn.cluster import KMeans
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics.pairwise import rbf_kernel

# sklearn — models
from sklearn.linear_model import LinearRegression


# --- Load data ---
# Dataset: California Housing (20,640 districts, 1990 census)
# Downloads and extracts from GitHub on first run, then reads from local CSV.

def load_housing_data():
    tarball_path = Path("datasets/housing.tgz")
    if not tarball_path.is_file():
        Path("datasets").mkdir(parents=True, exist_ok=True)
        url = "https://github.com/ageron/data/raw/main/housing.tgz"
        urllib.request.urlretrieve(url, tarball_path)
        with tarfile.open(tarball_path) as housing_tarball:
            housing_tarball.extractall(path="datasets", filter="data")
    return pd.read_csv(Path("datasets/housing/housing.csv"))


housing_full = load_housing_data()


# --- Stratified train/test split ---
# median_income is the strongest predictor (+0.69 correlation with target).
# A random split risks skewing its distribution by chance.
# Binning into 5 income categories and using stratify= ensures both sets
# are proportionally representative of the full income range.

housing_full["income_cat"] = pd.cut(
    housing_full["median_income"],
    bins=[0., 1.5, 3.0, 4.5, 6., np.inf],
    labels=[1, 2, 3, 4, 5]
)

strat_train_set, strat_test_set = train_test_split(
    housing_full, test_size=0.2, stratify=housing_full["income_cat"], random_state=42
)
for set_ in (strat_train_set, strat_test_set):
    set_.drop("income_cat", axis=1, inplace=True)

# Save test set to disk and remove from memory — not touched until final eval.
strat_test_set.to_csv("datasets/housing/test_set.csv", index=False)
del strat_test_set


# --- Separate features and target ---
# housing        → feature matrix (9 columns, includes 1 categorical)
# housing_labels → target vector  (median_house_value)

housing = strat_train_set.drop("median_house_value", axis=1)
housing_labels = strat_train_set["median_house_value"].copy()

missing = housing.isnull().sum()
missing = missing[missing > 0]
print(f"Training rows  : {housing.shape[0]:,}")
print(f"Features       : {housing.shape[1]}")
print(f"Missing values : {', '.join(f'{col} ({n})' for col, n in missing.items())}")
print(f"Columns        : {', '.join(housing.columns)}")

Training rows  : 16,512
Features       : 9
Missing values : total_bedrooms (168)


/var/folders/lz/wcp7zjcd1mx_25ng4w4vd2_m0000gn/T/ipykernel_19225/3282487858.py:50: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  housing_tarball.extractall(path="datasets")


In [3]:
# =============================================================================
# FULL PREPROCESSING PIPELINE
# =============================================================================
#
# All transformations are orchestrated by a single ColumnTransformer.
# It routes each column subset to the right pipeline in parallel,
# then concatenates outputs into one NumPy array: (16512, 24).
#
# FEATURE ENGINEERING — WHY EACH TRANSFORM EXISTS:
#
#   1. RATIO FEATURES (3 outputs)
#      Raw counts like total_rooms=6000 mean nothing alone.
#      Two districts with 6000 rooms are very different if one has 120
#      households (50 rooms/house = spacious) and the other has 3000
#      (2 rooms/house = cramped). Dividing gives density — the actual signal.
#
#        total_bedrooms / total_rooms  → bedroom ratio (how much is sleeping space?)
#        total_rooms / households      → rooms per house (spacious or cramped?)
#        population / households       → people per house (overcrowded?)
#
#   2. LOG TRANSFORM (5 outputs)
#      population, rooms, households, bedrooms, income all have long right tails.
#      A few extreme values (e.g., population 50,000 vs typical 300-500) dominate
#      the scale after StandardScaler — most districts get squashed into a tiny
#      range and become indistinguishable. log() compresses the highs and spreads
#      the lows, giving the model a more uniform spread to work with.
#
#      housing_median_age is skipped — already roughly uniform, no skew to fix.
#
#   3. CLUSTER SIMILARITY (10 outputs)
#      Raw lat/lon are just numbers — a linear model can't learn that
#      (37.77, -122.42) means "San Francisco" means "expensive."
#      KMeans finds 10 geographic centroids (Bay Area, LA, etc.), then
#      RBF kernel computes a smooth 0-to-1 "nearness" score for each:
#        K(x, c) = exp(-γ · ||x - c||²)  →  1.0 at centroid, ~0.0 far away
#      Now the model gets 10 features it can weight directly:
#        high Bay Area similarity → high price, high Central Valley → low price.
#
#   4. ONE-HOT ENCODING (5 outputs)
#      ocean_proximity has 5 categories with NO natural ordering.
#      OrdinalEncoder would assign integers (INLAND=1, ISLAND=2), implying
#      ISLAND is "more" than INLAND — false. OneHotEncoder gives each
#      category its own binary column, so no ordering is imposed.
#
#   5. REMAINDER — housing_median_age (1 output)
#      Already a meaningful unit (years), roughly uniform, numeric.
#      Just needs impute + scale — no special engineering.
#
# =============================================================================


# --- Custom transformer: geographic cluster similarity ---
# fit()  → KMeans on lat/lon → learns k centroids
# transform() → rbf_kernel(X, centroids) → similarity matrix

class ClusterSimilarity(BaseEstimator, TransformerMixin):
    def __init__(self, n_clusters=10, gamma=1.0, random_state=None):
        self.n_clusters = n_clusters
        self.gamma = gamma
        self.random_state = random_state

    def fit(self, X, y=None, sample_weight=None):
        self.kmeans_ = KMeans(self.n_clusters, random_state=self.random_state)
        self.kmeans_.fit(X, sample_weight=sample_weight)
        return self

    def transform(self, X):
        return rbf_kernel(X, self.kmeans_.cluster_centers_, gamma=self.gamma)

    def get_feature_names_out(self, names=None):
        return [f"Cluster {i} similarity" for i in range(self.n_clusters)]


# --- Ratio helper: wraps col[0]/col[1] as a sklearn transformer ---

def column_ratio(X):
    return X[:, [0]] / X[:, [1]]

def ratio_name(function_transformer, feature_names_in):
    return ["ratio"]

def ratio_pipeline():
    return make_pipeline(
        SimpleImputer(strategy="median"),
        FunctionTransformer(column_ratio, feature_names_out=ratio_name),
        StandardScaler(),
    )


# =============================================================================
# ColumnTransformer — the single entry point for all preprocessing
# =============================================================================
# Each row below: (name, pipeline, columns)
# All pipelines run in parallel on their column subsets, then concat → (n, 24)
#
#   remainder=default catches housing_median_age (the only unassigned column)
#   make_column_selector picks categorical columns dynamically by dtype

preprocessing = ColumnTransformer([

    # --- Ratio features: impute → divide col[0]/col[1] → scale ---
    ("bedrooms",         ratio_pipeline(), ["total_bedrooms", "total_rooms"]),
    ("rooms_per_house",  ratio_pipeline(), ["total_rooms", "households"]),
    ("people_per_house", ratio_pipeline(), ["population", "households"]),

    # --- Log transform: impute → log → scale (compresses right-skewed tails) ---
    ("log", make_pipeline(
        SimpleImputer(strategy="median"),
        FunctionTransformer(np.log, feature_names_out="one-to-one"),
        StandardScaler(),
    ), ["total_bedrooms", "total_rooms", "population", "households", "median_income"]),

    # --- Cluster similarity: KMeans centroids → RBF nearness scores ---
    ("geo", ClusterSimilarity(n_clusters=10, gamma=1., random_state=42),
     ["latitude", "longitude"]),

    # --- One-hot: nominal categories → binary columns (no false ordering) ---
    ("cat", make_pipeline(
        SimpleImputer(strategy="most_frequent"),
        OneHotEncoder(handle_unknown="ignore"),     # unseen categories → all zeros
    ), make_column_selector(dtype_include=object)),

],  # --- Remainder: housing_median_age → impute + scale (no special engineering) ---
    remainder=make_pipeline(SimpleImputer(strategy="median"), StandardScaler()),
)


housing_prepared = preprocessing.fit_transform(housing)

print(f"Output shape : {housing_prepared.shape}")
print(f"\nFeature names ({housing_prepared.shape[1]} total):")
for name in preprocessing.get_feature_names_out():
    print(f"  {name}")

Output shape : (16512, 24)

Feature names:
  bedrooms__ratio
  rooms_per_house__ratio
  people_per_house__ratio
  log__total_bedrooms
  log__total_rooms
  log__population
  log__households
  log__median_income
  geo__Cluster 0 similarity
  geo__Cluster 1 similarity
  geo__Cluster 2 similarity
  geo__Cluster 3 similarity
  geo__Cluster 4 similarity
  geo__Cluster 5 similarity
  geo__Cluster 6 similarity
  geo__Cluster 7 similarity
  geo__Cluster 8 similarity
  geo__Cluster 9 similarity
  cat__ocean_proximity_<1H OCEAN
  cat__ocean_proximity_INLAND
  cat__ocean_proximity_ISLAND
  cat__ocean_proximity_NEAR BAY
  cat__ocean_proximity_NEAR OCEAN
  remainder__housing_median_age

PIPELINE → FEATURE MAPPING

  Pipeline              Raw columns                    → Output features
  ─────────────────────────────────────────────────────────────────────
  bedrooms              total_bedrooms, total_rooms    → 1 ratio
  rooms_per_house       total_rooms, households        → 1 ratio
  people_per_ho

# Phase 2: Model Exploration — "Which algorithm works best?"
*Try progressively more powerful models to establish a performance ladder. Each model builds intuition about the data: Is the relationship linear? Does a single tree overfit? Does an ensemble help? We use cross-validated RMSE as the honest comparison metric — training error alone is misleading because the model has already seen that data.*

**Model progression:**
| Model | Complexity | Strengths | Weaknesses |
|-------|-----------|-----------|------------|
| LinearRegression | Lowest | Fast, interpretable, good baseline | Can't capture non-linear patterns |
| DecisionTree | Medium | Captures non-linear relationships | Memorizes training data (overfits) |
| RandomForest | High | Averages many trees → reduces overfitting | Slower, less interpretable |

In [ ]:
# =============================================================================
# LINEAR REGRESSION — Baseline ($70k RMSE)
# =============================================================================
#
# THE THEORY:
#   Linear Regression fits a hyperplane: y = w1·x1 + w2·x2 + ... + b
#   It finds weights (w) that minimize the sum of squared errors (OLS).
#   The relationship between features and target is assumed to be LINEAR —
#   doubling income → doubling the predicted price change.
#
# WHY START HERE:
#   No hyperparameters to tune — it's the simplest possible model.
#   If a complex model can't beat this, something is wrong upstream
#   (bad features, data leakage, etc.). Think of it as the "floor."
#
# WHAT CROSS-VALIDATION TELLS US:
#   Training RMSE alone is misleading — the model has seen that data.
#   10-fold CV splits the training set into 10 parts:
#     Train on 9 → evaluate on 1 → rotate 10 times → average.
#   This estimates how well the model generalizes to UNSEEN data.
#   The mean RMSE is our honest score; std tells us how stable it is
#   across different data splits.
#
# SCORING NOTE:
#   sklearn convention: higher = better. RMSE is a loss (lower = better),
#   so it's negated → "neg_root_mean_squared_error". We negate back.
# =============================================================================

lin_reg = make_pipeline(preprocessing, LinearRegression())
lin_reg.fit(housing, housing_labels)


# --- Cross-validated RMSE (10-fold) ---

rmse_scores = -cross_val_score(
    lin_reg, housing, housing_labels,
    scoring="neg_root_mean_squared_error", cv=10
)

print(f"CV RMSE scores : {rmse_scores.round(0)}")
print(f"Mean RMSE      : ${rmse_scores.mean():,.0f}")
print(f"Std RMSE       : ${rmse_scores.std():,.0f}")


# --- Full statistical summary ---
# describe() gives min/max/percentiles — reveals if any folds are outliers.
# A high std or wide min-max spread suggests the model is unstable.

print("\nCV RMSE distribution:")
pd.Series(rmse_scores).describe()


# --- Sample predictions vs actuals ---
# Spot-check: are errors randomly scattered, or systematically off?
# Large negative errors on expensive houses → model underpredicts high values
# (a sign that linear assumptions can't capture the full pattern).

sample = housing.iloc[:5]
predictions = lin_reg.predict(sample)

print("\nSample predictions vs actuals:")
pd.DataFrame({
    "actual":    housing_labels.iloc[:5].values,
    "predicted": predictions.round(0),
    "error":     (predictions - housing_labels.iloc[:5].values).round(0),
})

In [ ]:
# =============================================================================
# DECISION TREE REGRESSOR — Can non-linearity help? ($66k RMSE)
# =============================================================================
#
# THE THEORY:
#   A Decision Tree learns a series of if/else rules from the data:
#     "if median_income > 3.5 AND cluster_2_similarity > 0.7 → predict $320k"
#   At each node, it picks the feature and threshold that best splits the
#   data into groups with similar target values (minimizes variance).
#
# WHY TRY THIS:
#   Linear Regression assumes a straight-line relationship. Housing prices
#   are NOT linear — a house near the ocean in a rich neighborhood is worth
#   MORE than the sum of "near ocean" + "rich" would suggest. Decision Trees
#   can capture these interactions and non-linear patterns automatically.
#
# THE OVERFITTING PROBLEM:
#   An unrestricted Decision Tree will keep splitting until every training
#   sample has its own leaf node → training RMSE = $0 (perfect memorization).
#   But this is NOT learning — it's memorizing. The CV RMSE reveals the truth:
#     Training RMSE:  ~$0    (memorized)
#     CV RMSE:        ~$66k  (actual generalization ability)
#   This gap (training vs CV error) is the hallmark of overfitting.
#   The tree learned noise and quirks specific to the training data that
#   won't appear in new data.
#
# LESSON: Lower complexity (LinearReg) underfits. Unlimited complexity
# (DecisionTree) overfits. We need something in between → Random Forest.
# =============================================================================

from sklearn.tree import DecisionTreeRegressor

tree_reg = make_pipeline(preprocessing, DecisionTreeRegressor(random_state=42))
tree_reg.fit(housing, housing_labels)


# --- Cross-validated RMSE (10-fold) ---

rmse_scores = -cross_val_score(
    tree_reg, housing, housing_labels,
    scoring="neg_root_mean_squared_error", cv=10
)

print(f"CV RMSE scores : {rmse_scores.round(0)}")
print(f"Mean RMSE      : ${rmse_scores.mean():,.0f}")
print(f"Std RMSE       : ${rmse_scores.std():,.0f}")


# --- Full statistical summary ---

print("\nCV RMSE distribution:")
pd.Series(rmse_scores).describe()


# --- Sample predictions vs actuals ---
# Notice: errors are $0 — perfect memorization of training data.
# This looks great but is actually a red flag. The model will NOT
# perform this well on data it hasn't seen (as the CV RMSE confirms).

sample = housing.iloc[:5]
predictions = tree_reg.predict(sample)

print("\nSample predictions vs actuals:")
pd.DataFrame({
    "actual":    housing_labels.iloc[:5].values,
    "predicted": predictions.round(0),
    "error":     (predictions - housing_labels.iloc[:5].values).round(0),
})

In [ ]:
# =============================================================================
# RANDOM FOREST REGRESSOR — The power of ensembles ($47k RMSE)
# =============================================================================
#
# THE THEORY:
#   A single Decision Tree memorizes. But what if you trained 100 trees,
#   each on a slightly different version of the data, each seeing a random
#   subset of features — then averaged their predictions?
#
#   That's a Random Forest. It uses two sources of randomness:
#     1. BAGGING (Bootstrap Aggregating): each tree trains on a random
#        sample of rows (drawn with replacement from the training set).
#     2. RANDOM FEATURE SUBSETS: at each split, the tree can only choose
#        from a random subset of features (controlled by max_features).
#
#   This randomness makes each tree different. They each overfit in their
#   own unique way — but when you average 100 different overfitting patterns,
#   the individual errors cancel out. What remains is the true signal.
#
#   Analogy: ask 100 people to guess a cow's weight. Each guess is wrong,
#   but the AVERAGE of 100 guesses is remarkably close. The random errors
#   cancel; the shared signal (the actual weight) survives.
#
# WHY IT WORKS:
#   LinearRegression: too simple → underfits → $70k RMSE
#   DecisionTree:     too complex → overfits  → $66k RMSE
#   RandomForest:     many trees averaged → balances complexity → $47k RMSE
#
# n_estimators=100: number of trees. More trees = more stable predictions,
#   but diminishing returns past ~100 and each tree adds training time.
# =============================================================================

from sklearn.ensemble import RandomForestRegressor

forest_reg = make_pipeline(preprocessing, RandomForestRegressor(
    n_estimators=100, random_state=42
))
forest_reg.fit(housing, housing_labels)


# --- Cross-validated RMSE (10-fold) ---

rmse_scores = -cross_val_score(
    forest_reg, housing, housing_labels,
    scoring="neg_root_mean_squared_error", cv=10
)

print(f"CV RMSE scores : {rmse_scores.round(0)}")
print(f"Mean RMSE      : ${rmse_scores.mean():,.0f}")
print(f"Std RMSE       : ${rmse_scores.std():,.0f}")


# --- Full statistical summary ---

print("\nCV RMSE distribution:")
pd.Series(rmse_scores).describe()


# --- Sample predictions vs actuals ---
# Compare with DecisionTree's $0 errors. Random Forest errors are small
# but non-zero — it's not memorizing, it's genuinely learning patterns.
# This is healthier: imperfect on training data = better on new data.

sample = housing.iloc[:5]
predictions = forest_reg.predict(sample)

print("\nSample predictions vs actuals:")
pd.DataFrame({
    "actual":    housing_labels.iloc[:5].values,
    "predicted": predictions.round(0),
    "error":     (predictions - housing_labels.iloc[:5].values).round(0),
})

# Phase 3: Hyperparameter Tuning — "How do we squeeze more out of the winner?"
*The Random Forest won Phase 2, but it used default hyperparameters. Hyperparameters are settings YOU choose before training — the model can't learn them from data. Think of them as the "rules of the game" rather than the "moves within the game."*

**Two approaches:**
- **GridSearchCV** — you hand-pick specific values to try. Exhaustive but limited to your imagination.
- **RandomizedSearchCV** — you define a range, it samples random combinations. Explores wider, often finds surprises you wouldn't have thought to try.

*Both use cross-validation internally to honestly evaluate each combination.*

In [ ]:
# =============================================================================
# GRID SEARCH — Exhaustive hyperparameter tuning ($43.6k RMSE)
# =============================================================================
#
# THE THEORY:
#   Models have two types of settings:
#     - PARAMETERS: learned during training (tree split thresholds, weights)
#     - HYPERPARAMETERS: set by YOU before training. The model can't learn
#       these from data — they control HOW the model learns.
#
#   GridSearchCV answers: "Which combination of hyperparameters gives the
#   best generalization performance?" It tries EVERY combination you specify
#   and evaluates each with cross-validation.
#
# WHAT WE'RE TUNING:
#   1. preprocessing__geo__n_clusters: How finely to carve California's
#      geography. More clusters = more geographic features = the model can
#      distinguish more neighborhoods. But too many = overfitting to noise.
#
#   2. random_forest__max_features: How many features each tree can see at
#      each split. Lower = more diversity between trees (good for ensembles),
#      but too low = each tree is too restricted to learn well.
#
# WHY Pipeline() WITH NAMED STEPS:
#   make_pipeline auto-names steps ("randomforestregressor"), but Grid Search
#   parameter paths need to be readable. Pipeline() with explicit names gives:
#     "preprocessing__geo__n_clusters"  (clear: preprocessing → geo → n_clusters)
#   instead of:
#     "columntransformer__geo__n_clusters"  (what's a columntransformer?)
#
# TWO PARAM GRIDS (list of dicts):
#   Grid 1: [5, 8, 10] × [4, 6, 8] = 9 combos — fewer clusters, fewer features
#   Grid 2: [10, 15] × [6, 8, 10]  = 6 combos — more clusters, more features
#   Total: 15 combos × 3 folds = 45 training runs
#   Using two grids avoids wasting time on combos that don't make sense
#   (e.g., many clusters with very few features per split).
# =============================================================================

from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

full_pipeline = Pipeline([
    ("preprocessing", preprocessing),
    ("random_forest", RandomForestRegressor(random_state=42)),
])

param_grid = [
    {'preprocessing__geo__n_clusters': [5, 8, 10],
     'random_forest__max_features': [4, 6, 8]},
    {'preprocessing__geo__n_clusters': [10, 15],
     'random_forest__max_features': [6, 8, 10]},
]

grid_search = GridSearchCV(
    full_pipeline,
    param_grid,
    cv=3,
    scoring="neg_root_mean_squared_error",
)

grid_search.fit(housing, housing_labels)


# --- Best hyperparameters and score ---

print("Best hyperparameters:")
for param, value in grid_search.best_params_.items():
    name = param.split("__")[-1]
    print(f"  {name}: {value}")

best_rmse = -grid_search.best_score_
print(f"\nBest CV RMSE: ${best_rmse:,.0f}")
print(f"(vs default Random Forest: $47,038)")


# --- All results ranked by CV RMSE ---

results = pd.DataFrame(grid_search.cv_results_)
results["mean_rmse"] = -results["mean_test_score"]
results["std_rmse"] = results["std_test_score"]

display = results[[
    "param_preprocessing__geo__n_clusters",
    "param_random_forest__max_features",
    "mean_rmse", "std_rmse",
]].sort_values("mean_rmse")

display.columns = ["n_clusters", "max_features", "CV RMSE", "Std RMSE"]

print("\nAll combinations ranked by CV RMSE:")
display.round(0)

In [ ]:
# =============================================================================
# RANDOMIZED SEARCH — Smarter hyperparameter tuning
# =============================================================================
#
# THE THEORY:
#   GridSearch is exhaustive but limited — it only tries values you specify.
#   What if the best n_clusters is 37? You'd never know unless you put 37
#   in your grid.
#
#   RandomizedSearchCV solves this by SAMPLING from continuous distributions:
#     randint(3, 50) → can draw ANY integer from 3 to 49
#   You set a BUDGET (n_iter=10) — "try 10 random combinations."
#
# HOW IT WORKS (step by step):
#   1. DRAW 10 random (n_clusters, max_features) pairs from the distributions
#   2. For EACH combo, run 3-fold CV (train on 2 folds, eval on 1, rotate)
#   3. Rank all 10 combos by mean CV RMSE → pick the winner
#   4. Retrain the winner on ALL training data → best_estimator_
#   Total: 10 combos × 3 folds = 30 training runs
#
# WHY RANDOMIZED > GRID:
#   - GridSearch searched [5, 8, 10, 15] for n_clusters (4 values)
#   - RandomizedSearch searches [3, 49) (47 possible values)
#   - With the same budget, random search covers more ground
#   - Research shows: for most problems, random search finds equally good
#     results as grid search in a fraction of the time (Bergstra & Bengio, 2012)
#
# scipy.stats.randint(low, high):
#   Uniform distribution over integers [low, high). Each value equally likely.
#   random_state=42 makes the draws reproducible.
# =============================================================================

from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

param_distribs = {
    'preprocessing__geo__n_clusters': randint(low=3, high=50),
    'random_forest__max_features': randint(low=2, high=20),
}

rnd_search = RandomizedSearchCV(
    full_pipeline,
    param_distributions=param_distribs,
    n_iter=10,
    cv=3,
    scoring="neg_root_mean_squared_error",
    random_state=42,
)

rnd_search.fit(housing, housing_labels)


# --- Best hyperparameters and score ---

print("Best hyperparameters:")
for param, value in rnd_search.best_params_.items():
    name = param.split("__")[-1]
    print(f"  {name}: {value}")

best_rmse = -rnd_search.best_score_
print(f"\nBest CV RMSE: ${best_rmse:,.0f}")
print(f"(vs GridSearch best: $43,590)")
print(f"(vs default Random Forest: $47,038)")


# --- All results ranked by CV RMSE ---

results = pd.DataFrame(rnd_search.cv_results_)
results["mean_rmse"] = -results["mean_test_score"]
results["std_rmse"] = results["std_test_score"]

display_cols = results[[
    "param_preprocessing__geo__n_clusters",
    "param_random_forest__max_features",
    "mean_rmse", "std_rmse",
]].sort_values("mean_rmse")

display_cols.columns = ["n_clusters", "max_features", "CV RMSE", "Std RMSE"]

print("\nAll 10 random combinations ranked by CV RMSE:")
display_cols.round(0)

# Phase 4: Analyze the Best Model — "What did it learn?"
*Now that we've found the best hyperparameters, we look inside the model. Feature importances reveal which inputs actually drive predictions — and which ones are dead weight. This is both a sanity check (does the model agree with domain knowledge?) and a guide for future feature engineering.*

In [ ]:
# =============================================================================
# FEATURE IMPORTANCES — What drives California housing prices?
# =============================================================================
#
# Random Forest provides feature_importances_: a score for each feature
# measuring how much it contributed to reducing prediction error across
# all 100 trees. Scores sum to 1.0. Higher = more important.
#
# HOW IT'S CALCULATED:
#   Every time a tree splits on a feature, the prediction error decreases
#   by some amount. The importance of a feature = the total error reduction
#   from all splits using that feature, averaged across all 100 trees.
#
#   Example: if "median_income" is used in 80% of trees and each time
#   it produces a large error reduction → high importance (~0.19).
#   If "ocean_proximity_ISLAND" is rarely used and barely reduces error
#   → low importance (~0.00).
#
# WHY THIS MATTERS:
#   - Sanity check: does the model agree with our intuition?
#     (income SHOULD matter most for housing prices)
#   - Feature selection: if a feature has ~0 importance, we could drop it
#     to simplify the pipeline without losing accuracy.
#   - Debugging: if a feature you expected to matter has low importance,
#     it might be poorly engineered or redundant with another feature.
#
# best_estimator_ is the full pipeline (preprocessing + random_forest)
# already retrained on ALL training data with the best hyperparameters.
# =============================================================================

import matplotlib.pyplot as plt

# --- Extract the best model ---
# best_estimator_ includes the full pipeline (preprocessing + random_forest)
# Already retrained on all 16,512 rows with the winning hyperparameters.

final_model = rnd_search.best_estimator_
feature_importances = final_model["random_forest"].feature_importances_


# --- Sorted importances with feature names ---
# sorted(zip(importance, name)) pairs each score with its feature,
# then sorts descending. Simple pattern from the book.

sorted_importances = sorted(
    zip(feature_importances, final_model["preprocessing"].get_feature_names_out()),
    reverse=True
)

print("Feature importances (descending):")
print("-" * 55)
for importance, name in sorted_importances:
    bar = "█" * int(importance * 100)
    print(f"  {name:40s} {importance:.4f}  {bar}")


# --- Horizontal bar chart ---
# Visual ranking makes it easy to spot the dominant features at a glance
# and identify the "long tail" of low-importance features.

feature_names = final_model["preprocessing"].get_feature_names_out()
sorted_idx = feature_importances.argsort()

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(range(len(feature_importances)), feature_importances[sorted_idx], align="center")
ax.set_yticks(range(len(feature_names)))
ax.set_yticklabels(feature_names[sorted_idx])
ax.set_xlabel("Feature Importance")
ax.set_title("Random Forest — Feature Importances (Best Model)")
plt.tight_layout()
plt.show()

# Phase 5: Evaluate on Test Set — "How will it perform in the real world?"
*This is the moment of truth. The test set has been locked away since Cell 0 — the model has NEVER seen this data. No tuning, no adjustments, no second chances. This single number is our best estimate of how the model will perform on genuinely new districts.*

*If the test RMSE is close to the CV RMSE → our cross-validation was honest and the model generalizes well. If it's much worse → we may have overfit during the tuning process (yes, tuning itself can overfit if you try too many combinations).*

In [ ]:
# =============================================================================
# FINAL EVALUATION ON TEST SET
# =============================================================================
#
# THE THEORY:
#   Throughout this notebook we used CROSS-VALIDATION on the training set
#   to estimate generalization. But CV scores were used to CHOOSE models
#   and hyperparameters — so even the CV estimate is slightly optimistic.
#
#   The test set is the only truly unbiased estimate. It was:
#     - Split off at the very beginning (Cell 0)
#     - Saved to disk and deleted from memory
#     - Never used for training, feature selection, or hyperparameter tuning
#
#   This is like a final exam you've never seen the questions for.
#   Your CV scores were practice tests — helpful, but the final exam
#   is the real measure.
#
# CONFIDENCE INTERVAL:
#   A single RMSE number doesn't tell the full story. We compute a 95%
#   confidence interval: "we're 95% confident the true generalization
#   error falls between $X and $Y." This accounts for the fact that our
#   test set is a finite sample — a different 20% split would give a
#   slightly different RMSE.
#
# IMPORTANT:
#   If the test RMSE is bad, you do NOT go back and tune more.
#   That would be "teaching to the test" — the test set would no longer
#   be unbiased. You accept the result and move on.
# =============================================================================

from sklearn.metrics import mean_squared_error
from scipy import stats

# --- Load the test set (untouched since Cell 0) ---

strat_test_set = pd.read_csv("datasets/housing/test_set.csv")

X_test = strat_test_set.drop("median_house_value", axis=1)
y_test = strat_test_set["median_house_value"].copy()


# --- Predict using the best model ---
# final_model = rnd_search.best_estimator_ (set in Phase 4)
# This is the full pipeline: preprocessing + RandomForest with best hyperparams.
# preprocessing.transform() is called automatically — it uses the parameters
# learned from training data (medians, cluster centroids, etc.), NOT test data.

final_predictions = final_model.predict(X_test)


# --- Test RMSE ---

final_rmse = mean_squared_error(y_test, final_predictions, squared=False)
print(f"Test RMSE: ${final_rmse:,.0f}")
print(f"(vs best CV RMSE: ${-rnd_search.best_score_:,.0f})")


# --- 95% Confidence Interval ---
# Uses a t-distribution to account for the finite test set size.
# Squared errors are the individual data points; we compute a confidence
# interval on their mean, then take the square root to get RMSE bounds.

squared_errors = (final_predictions - y_test) ** 2
confidence = 0.95

ci = np.sqrt(stats.t.interval(
    confidence,
    len(squared_errors) - 1,
    loc=squared_errors.mean(),
    scale=stats.sem(squared_errors),
))

print(f"\n95% Confidence Interval: [${ci[0]:,.0f}, ${ci[1]:,.0f}]")
print(f"  → We're 95% confident the true error is in this range")


# --- Sample predictions vs actuals ---

sample_idx = np.random.RandomState(42).choice(len(X_test), 10, replace=False)

print("\nSample predictions vs actuals (10 random test districts):")
pd.DataFrame({
    "actual":    y_test.iloc[sample_idx].values,
    "predicted": final_predictions[sample_idx].round(0),
    "error":     (final_predictions[sample_idx] - y_test.iloc[sample_idx].values).round(0),
})